# Split-Step LSTM Classifier
Binary classifier: **1 = split step**, **0 = not a split step**.

## Imports & Configuration

In [ ]:
import json
import pathlib
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

In [ ]:
# MediaPipe landmark indices used as features
FEATURE_LANDMARK_NAMES = [
    "left_ankle",  "left_heel",  "left_foot_index",
    "right_ankle", "right_heel", "right_foot_index",
    "left_hip",    "right_hip",
    "left_shoulder", "right_shoulder",
    "left_elbow",  "right_elbow",
    "left_wrist",  "right_wrist",
]

N_FEATURES = len(FEATURE_LANDMARK_NAMES) * 3   # x, y, z per landmark
SEQ_LEN    = 30                                 # frames per window (~1 s at 30 fps)

print(f"Features per frame : {N_FEATURES}")
print(f"Sequence length    : {SEQ_LEN} frames")

## Load Dataset from CSV

In [ ]:
def load_csv_dataset(csv_path: str) -> Tuple[List[np.ndarray], List[int]]:
    """Load features and binary labels from a labeled pose CSV."""
    df = pd.read_csv(csv_path)

    feature_cols = []
    for name in FEATURE_LANDMARK_NAMES:
        feature_cols += [f"{name}_world_x", f"{name}_world_y", f"{name}_world_z"]

    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    if "label" not in df.columns:
        raise ValueError("'label' column not found — run csv_labeler.py first.")

    X = df[feature_cols].values.astype(np.float32)
    y = df["label"].values.astype(int).tolist()
    return [X[i] for i in range(len(X))], y


CSV_PATH = "assets/pose1_labeled.csv"
feature_frames, label_frames = load_csv_dataset(CSV_PATH)

total    = len(label_frames)
positive = sum(label_frames)
print(f"Loaded {total} frames from {CSV_PATH}")
print(f"Split-step frames : {positive} ({positive/total*100:.1f}%)")
print(f"Non-split frames  : {total - positive} ({(total-positive)/total*100:.1f}%)")

## Dataset

In [ ]:
def build_windows(
    feature_frames: List[np.ndarray],
    label_frames:   List[int],
    seq_len:        int = SEQ_LEN,
) -> Tuple[List[np.ndarray], List[int]]:
    """Slice a continuous recording into overlapping sliding windows."""
    seqs, labels = [], []
    for i in range(seq_len - 1, len(feature_frames)):
        window = np.stack(feature_frames[i - seq_len + 1 : i + 1])
        seqs.append(window)
        labels.append(label_frames[i])
    return seqs, labels


class SplitStepDataset(Dataset):
    """Each sample: (seq [SEQ_LEN, N_FEATURES], label [0 or 1])."""

    def __init__(self, seqs: List[np.ndarray], labels: List[int]):
        self.X = [torch.from_numpy(s) for s in seqs]
        self.y = [torch.tensor(l, dtype=torch.float32) for l in labels]

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]

## Train / Test Split

In [ ]:
seqs, labels = build_windows(feature_frames, label_frames, SEQ_LEN)
n         = len(seqs)
train_end = int(n * 0.75)

train_seqs,  train_labels = seqs[:train_end],  labels[:train_end]
test_seqs,   test_labels  = seqs[train_end:],  labels[train_end:]

print(f"Total windows : {n}")
print(f"Train         : {len(train_seqs)} ({len(train_seqs)/n*100:.0f}%)  — {sum(train_labels)} positive")
print(f"Test          : {len(test_seqs)}  ({len(test_seqs)/n*100:.0f}%)  — {sum(test_labels)} positive")

## Model Definition

In [ ]:
MODEL_PATH = "split_step_lstm.pt"
EPOCHS     = 40
BATCH_SIZE = 32
LR         = 1e-3
VAL_SPLIT  = 0.15   # fraction of training data used for validation
THRESHOLD  = 0.5

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

val_start = int(len(train_seqs) * (1 - VAL_SPLIT))
t_seqs,  t_labels = train_seqs[:val_start],  train_labels[:val_start]
v_seqs,  v_labels = train_seqs[val_start:],  train_labels[val_start:]

train_dl = DataLoader(SplitStepDataset(t_seqs, t_labels), batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_dl   = DataLoader(SplitStepDataset(v_seqs, v_labels), batch_size=BATCH_SIZE, shuffle=False)

print(f"Training : {len(t_seqs)}  |  Validation : {len(v_seqs)}")

model     = SplitStepLSTM().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)

best_val_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_dl:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_dl:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds    = (torch.sigmoid(model(X_batch)) >= THRESHOLD).float()
            correct += (preds == y_batch).sum().item()
            total   += y_batch.size(0)

    val_acc = correct / total if total else 0.0
    scheduler.step(1 - val_acc)
    print(f"Epoch {epoch:3d}/{EPOCHS}  loss={running_loss / len(train_dl):.4f}  val_acc={val_acc:.3f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)

print(f"\nBest val_acc={best_val_acc:.3f} — model saved -> {MODEL_PATH}")

## Test Evaluation

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()

test_dl    = DataLoader(SplitStepDataset(test_seqs, test_labels), batch_size=BATCH_SIZE, shuffle=False)
all_preds  = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_dl:
        probs  = torch.sigmoid(model(X_batch.to(device))).cpu()
        preds  = (probs >= THRESHOLD).float()
        all_preds.extend(preds.tolist())
        all_labels.extend(y_batch.tolist())

all_preds  = torch.tensor(all_preds)
all_labels = torch.tensor(all_labels)

tp = ((all_preds == 1) & (all_labels == 1)).sum().item()
fp = ((all_preds == 1) & (all_labels == 0)).sum().item()
tn = ((all_preds == 0) & (all_labels == 0)).sum().item()
fn = ((all_preds == 0) & (all_labels == 1)).sum().item()

accuracy  = (tp + tn) / len(all_labels)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"Test set results ({len(all_labels)} windows)")
print(f"  Accuracy  : {accuracy:.3f}")
print(f"  Precision : {precision:.3f}")
print(f"  Recall    : {recall:.3f}")
print(f"  F1        : {f1:.3f}")
print(f"\n  TP={tp}  FP={fp}  TN={tn}  FN={fn}")

## Training

In [ ]:
MODEL_PATH = "split_step_lstm.pt"
EPOCHS     = 40
BATCH_SIZE = 32
LR         = 1e-3
VAL_SPLIT  = 0.15
THRESHOLD  = 0.5   # sigmoid probability threshold

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

seqs, labels = build_windows(feature_frames, label_frames, SEQ_LEN)
print(f"Windows: {len(seqs)}  |  positive: {sum(labels)} ({sum(labels)/len(labels)*100:.1f}%)")

split    = int(len(seqs) * (1 - VAL_SPLIT))
train_ds = SplitStepDataset(seqs[:split],  labels[:split])
val_ds   = SplitStepDataset(seqs[split:],  labels[split:])
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

model     = SplitStepLSTM().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)

best_val_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_dl:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_dl:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds    = (torch.sigmoid(model(X_batch)) >= THRESHOLD).float()
            correct += (preds == y_batch).sum().item()
            total   += y_batch.size(0)

    val_acc = correct / total if total else 0.0
    scheduler.step(1 - val_acc)
    print(f"Epoch {epoch:3d}/{EPOCHS}  loss={running_loss / len(train_dl):.4f}  val_acc={val_acc:.3f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)

print(f"\nBest val_acc={best_val_acc:.3f} — model saved -> {MODEL_PATH}")

## Inference — Predictor

In [ ]:
class Predictor:
    """
    Rolling-window binary predictor.
    Usage: predictor = Predictor(); is_split, conf = predictor.update(feature_vec)
    """

    def __init__(
        self,
        model_path: str   = "split_step_lstm.pt",
        seq_len:    int   = SEQ_LEN,
        threshold:  float = THRESHOLD,
        device:     Optional[str] = None,
    ):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device    = device
        self.seq_len   = seq_len
        self.threshold = threshold
        self.buffer: List[np.ndarray] = []

        self.model = SplitStepLSTM().to(device)
        self.model.load_state_dict(
            torch.load(model_path, map_location=device, weights_only=True)
        )
        self.model.eval()

    def update(self, features: np.ndarray) -> Tuple[bool, float]:
        """
        Feed one (N_FEATURES,) array.
        Returns (is_split_step, confidence) once the buffer is full;
        returns (False, 0.0) while warming up.
        """
        self.buffer.append(features)
        if len(self.buffer) > self.seq_len:
            self.buffer.pop(0)
        if len(self.buffer) < self.seq_len:
            return False, 0.0

        x = torch.from_numpy(np.stack(self.buffer)).unsqueeze(0).to(self.device)
        with torch.no_grad():
            prob = torch.sigmoid(self.model(x)).item()
        return prob >= self.threshold, prob

## Persistence (JSON)

In [ ]:
def save_dataset(
    feature_frames: List[np.ndarray],
    label_frames:   List[int],
    path:           str,
) -> None:
    payload = {
        "features": [f.tolist() for f in feature_frames],
        "labels":   label_frames,
    }
    pathlib.Path(path).write_text(json.dumps(payload))


def load_dataset(path: str) -> Tuple[List[np.ndarray], List[int]]:
    payload = json.loads(pathlib.Path(path).read_text())
    features = [np.array(f, dtype=np.float32) for f in payload["features"]]
    return features, payload["labels"]